In [0]:
%sql
USE CATALOG ecommerce;
USE SCHEMA project;

In [0]:
%sql
SELECT
    current_catalog() AS catalog,
    current_schema() AS schema;

In [0]:
customers_raw_df = spark.table("ecommerce.project.customers_raw")

products_raw_df = spark.table("ecommerce.project.products_raw")

orders_raw_df = spark.table("ecommerce.project.orders_raw")

display(customers_raw_df)
display(products_raw_df)
display(orders_raw_df)


In [0]:
print("Customers:", customers_raw_df.count())
print("Products:", products_raw_df.count())
print("Orders:", orders_raw_df.count())

Cleaning customers data

In [0]:
customers_clean_df = customers_raw_df.dropDuplicates(["customer_id"])
print("Before:", customers_raw_df.count())
print("After duplicate removal:", customers_clean_df.count())

In [0]:
customers_clean_df = customers_clean_df.filter(
    customers_clean_df.customer_id.isNotNull()
)
print("After null customer_id removal:", customers_clean_df.count())

In [0]:
from pyspark.sql.functions import trim

customers_clean_df = (
    customers_clean_df
    .withColumn("customer_id", trim("customer_id"))
    .withColumn("customer_name", trim("customer_name"))
    .withColumn("city", trim("city"))
    .withColumn("state", trim("state"))
)
display(customers_clean_df)

In [0]:
(
    customers_clean_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("customers_clean")
)

In [0]:
%sql
select * from customers_clean

In [0]:
products_df = products_raw_df.dropDuplicates(["product_id"])
print("Before:", products_raw_df.count())
print("After duplicate removal:", products_df.count())

In [0]:
from pyspark.sql.functions import *

In [0]:
products_df.filter(
    col("product_id").isNull()
).show()

In [0]:
products_df = products_df.filter(
    col("product_id").isNotNull()
)
print("After null product_id removal:", products_df.count())

In [0]:
products_df = products_df.filter(
    col("price") > 0
)
print( products_df.count())

In [0]:
display(products_df)

In [0]:
products_df = products_df.withColumn(
    "category",
    when(col("category") == "Electronic", "Electronics")
    .otherwise(col("category"))
)
display(products_df)

In [0]:
from pyspark.sql.functions import trim

products_df = (
    products_df
    .withColumn("product_id", trim(col("product_id")))
    .withColumn("product_name", trim(col("product_name")))
    .withColumn("category", trim(col("category")))
)

In [0]:
display(products_df)

In [0]:
(
    products_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("products_clean")
)

In [0]:
%sql
SELECT COUNT(*) AS product_count
FROM products_clean;

In [0]:
orders_raw_df = spark.table("ecommerce.project.orders_raw")

display(orders_raw_df)

In [0]:
orders_raw_df.printSchema()

In [0]:
orders_df = orders_raw_df.dropDuplicates(["order_id"])

print("Before:", orders_raw_df.count())
print("After duplicate removal:", orders_df.count())

In [0]:
from pyspark.sql.functions import *

In [0]:
orders_df = orders_df.filter(
    col("customer_id").isNotNull()
)

In [0]:
orders_df = orders_df.filter(
    col("product_id").isNotNull()
)
display(orders_df)


In [0]:
orders_df.printSchema()

In [0]:
orders_raw_df = spark.table("ecommerce.project.orders_raw")

display(orders_raw_df)

In [0]:
from pyspark.sql.functions import col, count, expr, trim

orders_df = orders_raw_df.dropDuplicates(["order_id"])

print("After duplicate removal:", orders_df.count())

In [0]:
orders_df = orders_df.filter(
    col("customer_id").isNotNull()
    & col("product_id").isNotNull()
)

print("After null key removal:", orders_df.count())

In [0]:
orders_df = orders_df.withColumn(
    "order_date",
    expr("try_to_date(order_date, 'yyyy-MM-dd')")
)
display(orders_df)

In [0]:
orders_df = orders_df.filter(
    col("order_date").isNotNull()
)

print("After invalid date removal:", orders_df.count())

In [0]:
orders_df = orders_df.filter(
    col("quantity") > 0
)

print("After quantity validation:", orders_df.count())

In [0]:
orders_df = (
    orders_df
    .withColumn("order_id", trim(col("order_id")))
    .withColumn("customer_id", trim(col("customer_id")))
    .withColumn("product_id", trim(col("product_id")))
)

In [0]:
(
    orders_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("ecommerce.project.orders_clean")
)

In [0]:
%sql
drop table ecommerce.project.orders

In [0]:
customers_raw_df = spark.table("ecommerce.project.customers_clean")

products_raw_df = spark.table("ecommerce.project.products_clean")

orders_raw_df = spark.table("ecommerce.project.orders_clean")

print("Customers:", customers_raw_df.count())
print("Products:", products_raw_df.count())
print("Orders:", orders_raw_df.count())